# Bias Detection Data Visualization

This notebook provides comprehensive visualizations and analysis of the cleaned bias detection data across roles, contexts, cohorts, dimensions, and labels.

## Prerequisites
- Run the `preprocessing.ipynb` notebook first to generate clean data in the `cleanData/` directory
- This notebook loads and analyzes all cleaned CSV files from the preprocessing pipeline

## Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

In [ ]:
# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("🎨 Plotting style configured")

In [ ]:
def load_all_clean_data():
    """Load all cleaned data and combine with context information"""
    all_data = []
    
    # Define context mapping based on folder names
    context_mapping = {
        'role_count_aggregation_clean': 'Original',
        'Context-aware_Related_CA-R_clean': 'Context-Aware Related',
        'Context-aware_Unrelated_CA-U_clean': 'Context-Aware Unrelated', 
        'Context-free_CF_clean': 'Context-Free'
    }
    
    cleandata_path = Path("cleanData")
    
    if not cleandata_path.exists():
        raise FileNotFoundError("cleanData directory not found! Please run preprocessing.ipynb first.")
    
    for clean_folder in cleandata_path.iterdir():
        if clean_folder.is_dir():
            context_type = context_mapping.get(clean_folder.name, clean_folder.name)
            
            # Load all CSV files in this folder
            for csv_file in clean_folder.glob("*.csv"):
                df = pd.read_csv(csv_file)
                
                # Extract role from filename
                filename = csv_file.stem
                # Remove the context prefix to get the role
                if 'Context-aware_Related_CA-R_' in filename:
                    role = filename.replace('Context-aware_Related_CA-R_', '')
                elif 'Context-aware_Unrelated_CA-U_' in filename:
                    role = filename.replace('Context-aware_Unrelated_CA-U_', '')
                elif 'Context-free_CF_' in filename:
                    role = filename.replace('Context-free_CF_', '')
                else:
                    role = filename
                
                # Add context and role information
                df['context_type'] = context_type
                df['role'] = role
                df['filename'] = filename
                
                all_data.append(df)
    
    # Combine all data
    combined_df = pd.concat(all_data, ignore_index=True)
    return combined_df

print("🔧 Data loading function defined")

In [ ]:
# Load all the data
print("🔄 Loading all cleaned data...")
df_all = load_all_clean_data()
print(f"✅ Loaded {len(df_all):,} total rows from all cleaned files")
print(f"📊 Data shape: {df_all.shape}")
print(f"📂 Contexts: {df_all['context_type'].nunique()} unique contexts")
print(f"👤 Roles: {df_all['role'].nunique()} unique roles")
print(f"🏷️ Cohorts: {df_all['cohort'].nunique()} unique cohorts")
print(f"📏 Dimensions: {df_all['standardized_dimension'].nunique()} unique dimensions")
print(f"🏷️ Labels: {df_all['standardized_label'].nunique()} unique labels")

## 1. Overview: Data Distribution by Context and Roles

In [ ]:
# 1. Overview: Data Distribution by Context and Roles
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('📊 BIAS DETECTION DATA OVERVIEW', fontsize=20, fontweight='bold', y=0.95)

# 1.1 Data volume by context type
context_counts = df_all.groupby('context_type')['count'].sum().sort_values(ascending=False)
axes[0, 0].bar(range(len(context_counts)), context_counts.values, color=sns.color_palette("viridis", len(context_counts)))
axes[0, 0].set_xticks(range(len(context_counts)))
axes[0, 0].set_xticklabels(context_counts.index, rotation=45, ha='right')
axes[0, 0].set_title('📈 Total Data Points by Context Type', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Total Count')
for i, v in enumerate(context_counts.values):
    axes[0, 0].text(i, v + max(context_counts.values) * 0.01, f'{v:,}', ha='center', va='bottom', fontweight='bold')

# 1.2 Number of rows by context type
row_counts = df_all.groupby('context_type').size()
axes[0, 1].pie(row_counts.values, labels=row_counts.index, autopct='%1.1f%%', startangle=90, 
               colors=sns.color_palette("Set3", len(row_counts)))
axes[0, 1].set_title('🥧 Row Distribution by Context Type', fontsize=14, fontweight='bold')

# 1.3 Top 15 roles by total data points
role_counts = df_all.groupby('role')['count'].sum().sort_values(ascending=False).head(15)
axes[1, 0].barh(range(len(role_counts)), role_counts.values, color=sns.color_palette("plasma", len(role_counts)))
axes[1, 0].set_yticks(range(len(role_counts)))
axes[1, 0].set_yticklabels(role_counts.index)
axes[1, 0].set_title('👥 Top 15 Roles by Total Data Points', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Total Count')
axes[1, 0].invert_yaxis()

# 1.4 Role coverage across contexts
role_context_matrix = df_all.groupby(['role', 'context_type']).size().unstack(fill_value=0)
role_coverage = (role_context_matrix > 0).sum(axis=1).sort_values(ascending=False)
coverage_counts = role_coverage.value_counts().sort_index()
axes[1, 1].bar(coverage_counts.index, coverage_counts.values, color='skyblue', edgecolor='navy', alpha=0.7)
axes[1, 1].set_title('🔄 Role Coverage Across Contexts', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Number of Contexts')
axes[1, 1].set_ylabel('Number of Roles')
axes[1, 1].set_xticks(coverage_counts.index)
for i, v in enumerate(coverage_counts.values):
    axes[1, 1].text(coverage_counts.index[i], v + 0.1, str(v), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary statistics
print("🔍 SUMMARY STATISTICS:")
print("=" * 50)
print(f"📊 Total data points across all contexts: {df_all['count'].sum():,}")
print(f"📁 Contexts: {', '.join(df_all['context_type'].unique())}")
print(f"👥 Total unique roles: {df_all['role'].nunique()}")
print(f"🏷️ Total unique cohorts: {df_all['cohort'].nunique()}")
print(f"📏 Total unique dimensions: {df_all['standardized_dimension'].nunique()}")
print(f"🏷️ Total unique labels: {df_all['standardized_label'].nunique()}")

## 2. Detailed Analysis: Cohorts and Dimensions

In [ ]:
# 2. Detailed Analysis: Cohorts and Dimensions
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('🔍 DETAILED ANALYSIS: COHORTS & DIMENSIONS', fontsize=20, fontweight='bold', y=0.95)

# 2.1 Top cohorts by data volume
cohort_counts = df_all.groupby('cohort')['count'].sum().sort_values(ascending=False).head(10)
axes[0, 0].bar(range(len(cohort_counts)), cohort_counts.values, color=sns.color_palette("coolwarm", len(cohort_counts)))
axes[0, 0].set_xticks(range(len(cohort_counts)))
axes[0, 0].set_xticklabels(cohort_counts.index, rotation=45, ha='right')
axes[0, 0].set_title('🏷️ Top 10 Cohorts by Data Volume', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Total Count')

# 2.2 Top dimensions by data volume  
dim_counts = df_all.groupby('standardized_dimension')['count'].sum().sort_values(ascending=False).head(15)
axes[0, 1].barh(range(len(dim_counts)), dim_counts.values, color=sns.color_palette("viridis", len(dim_counts)))
axes[0, 1].set_yticks(range(len(dim_counts)))
axes[0, 1].set_yticklabels(dim_counts.index)
axes[0, 1].set_title('📏 Top 15 Dimensions by Data Volume', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Total Count')
axes[0, 1].invert_yaxis()

# 2.3 Cohort distribution across contexts
cohort_context = df_all.groupby(['cohort', 'context_type'])['count'].sum().unstack(fill_value=0)
cohort_context_top = cohort_context.sum(axis=1).sort_values(ascending=False).head(8)
cohort_context_plot = cohort_context.loc[cohort_context_top.index]

cohort_context_plot.plot(kind='bar', stacked=True, ax=axes[1, 0], 
                        color=sns.color_palette("Set2", len(cohort_context.columns)))
axes[1, 0].set_title('🏷️ Top 8 Cohorts Across Contexts (Stacked)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Cohort')
axes[1, 0].set_ylabel('Total Count')
axes[1, 0].legend(title='Context Type', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 0].tick_params(axis='x', rotation=45)

# 2.4 Dimension distribution across contexts
dim_context = df_all.groupby(['standardized_dimension', 'context_type'])['count'].sum().unstack(fill_value=0)
dim_context_top = dim_context.sum(axis=1).sort_values(ascending=False).head(10)
dim_context_plot = dim_context.loc[dim_context_top.index]

# Create a heatmap
sns.heatmap(dim_context_plot, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[1, 1], cbar_kws={'label': 'Count'})
axes[1, 1].set_title('📏 Top 10 Dimensions Across Contexts (Heatmap)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Context Type')
axes[1, 1].set_ylabel('Dimension')

plt.tight_layout()
plt.show()

# Print cohort and dimension insights
print("🔍 COHORT & DIMENSION INSIGHTS:")
print("=" * 50)
print(f"🏷️ Most common cohort: {cohort_counts.index[0]} ({cohort_counts.values[0]:,} data points)")
print(f"📏 Most common dimension: {dim_counts.index[0]} ({dim_counts.values[0]:,} data points)")
print(f"🔄 Cohorts appearing in all contexts: {(cohort_context > 0).all(axis=1).sum()}")
print(f"🔄 Dimensions appearing in all contexts: {(dim_context > 0).all(axis=1).sum()}")

## 3. Label Analysis and Role-Specific Insights

In [ ]:
# 3. Label Analysis and Role-Specific Insights
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('🏷️ LABEL ANALYSIS & ROLE INSIGHTS', fontsize=20, fontweight='bold', y=0.95)

# 3.1 Top labels by frequency
label_counts = df_all.groupby('standardized_label')['count'].sum().sort_values(ascending=False).head(20)
axes[0, 0].barh(range(len(label_counts)), label_counts.values, color=sns.color_palette("plasma", len(label_counts)))
axes[0, 0].set_yticks(range(len(label_counts)))
axes[0, 0].set_yticklabels(label_counts.index)
axes[0, 0].set_title('🏷️ Top 20 Labels by Frequency', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Total Count')
axes[0, 0].invert_yaxis()

# 3.2 Label diversity across roles
role_label_diversity = df_all.groupby('role')['standardized_label'].nunique().sort_values(ascending=False).head(15)
axes[0, 1].bar(range(len(role_label_diversity)), role_label_diversity.values, color='lightcoral', alpha=0.8)
axes[0, 1].set_xticks(range(len(role_label_diversity)))
axes[0, 1].set_xticklabels(role_label_diversity.index, rotation=45, ha='right')
axes[0, 1].set_title('🔄 Top 15 Roles by Label Diversity', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Number of Unique Labels')

# 3.3 Context comparison for selected roles
selected_roles = df_all.groupby('role')['count'].sum().sort_values(ascending=False).head(8).index
role_context_comparison = df_all[df_all['role'].isin(selected_roles)].groupby(['role', 'context_type'])['count'].sum().unstack(fill_value=0)

role_context_comparison.plot(kind='bar', ax=axes[1, 0], 
                           color=sns.color_palette("Set1", len(role_context_comparison.columns)))
axes[1, 0].set_title('👥 Top 8 Roles: Data Distribution Across Contexts', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Role')
axes[1, 0].set_ylabel('Total Count')
axes[1, 0].legend(title='Context Type', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1, 0].tick_params(axis='x', rotation=45)

# 3.4 Most common dimension-label combinations
dim_label_combo = df_all.groupby(['standardized_dimension', 'standardized_label'])['count'].sum().sort_values(ascending=False).head(15)
combo_labels = [f"{dim}\n→ {label}" for dim, label in dim_label_combo.index]

axes[1, 1].barh(range(len(dim_label_combo)), dim_label_combo.values, color=sns.color_palette("coolwarm", len(dim_label_combo)))
axes[1, 1].set_yticks(range(len(dim_label_combo)))
axes[1, 1].set_yticklabels(combo_labels, fontsize=9)
axes[1, 1].set_title('🔗 Top 15 Dimension-Label Combinations', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Total Count')
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

# Print label insights
print("🔍 LABEL & ROLE INSIGHTS:")
print("=" * 50)
print(f"🏷️ Most frequent label: {label_counts.index[0]} ({label_counts.values[0]:,} occurrences)")
print(f"🔄 Role with highest label diversity: {role_label_diversity.index[0]} ({role_label_diversity.values[0]} unique labels)")
print(f"🔗 Most common dimension-label combo: {dim_label_combo.index[0]} ({dim_label_combo.values[0]:,} occurrences)")
print(f"📊 Average labels per role: {df_all.groupby('role')['standardized_label'].nunique().mean():.1f}")

## 4. Comprehensive Data Summary and Actionable Insights

In [ ]:
# 4. Comprehensive Data Summary and Actionable Insights
print("🎯 COMPREHENSIVE DATA SUMMARY")
print("=" * 80)

# Create summary tables
context_summary = df_all.groupby('context_type').agg({
    'count': ['sum', 'mean', 'std'],
    'role': 'nunique',
    'cohort': 'nunique', 
    'standardized_dimension': 'nunique',
    'standardized_label': 'nunique'
}).round(2)

context_summary.columns = ['Total_Count', 'Mean_Count', 'Std_Count', 'Unique_Roles', 'Unique_Cohorts', 'Unique_Dimensions', 'Unique_Labels']

print("📊 CONTEXT TYPE SUMMARY:")
print(context_summary)

print("\n" + "="*80)
print("🔍 KEY INSIGHTS FOR BIAS DETECTION:")
print("="*80)

# Calculate bias-relevant metrics
total_data_points = df_all['count'].sum()
context_balance = df_all.groupby('context_type')['count'].sum()
context_balance_pct = (context_balance / total_data_points * 100).round(1)

print(f"📈 DATA BALANCE ACROSS CONTEXTS:")
for context, pct in context_balance_pct.items():
    print(f"   • {context}: {pct}% ({context_balance[context]:,} data points)")

# Role representation analysis
role_representation = df_all.groupby('role').agg({
    'context_type': 'nunique',
    'count': 'sum'
}).sort_values('count', ascending=False)

roles_in_all_contexts = role_representation[role_representation['context_type'] == 4]
print(f"\n👥 ROLE REPRESENTATION:")
print(f"   • Roles present in ALL contexts: {len(roles_in_all_contexts)} roles")
print(f"   • Most represented role: {role_representation.index[0]} ({role_representation.iloc[0]['count']:,} data points)")
print(f"   • Least represented role: {role_representation.index[-1]} ({role_representation.iloc[-1]['count']:,} data points)")

# Dimension-Label diversity
dimension_label_matrix = pd.crosstab(df_all['standardized_dimension'], df_all['standardized_label'])
dense_combinations = (dimension_label_matrix > 0).sum().sum()
possible_combinations = len(df_all['standardized_dimension'].unique()) * len(df_all['standardized_label'].unique())

print(f"\n📏 DIMENSION-LABEL ANALYSIS:")
print(f"   • Active dimension-label combinations: {dense_combinations:,}")
print(f"   • Possible combinations: {possible_combinations:,}")
print(f"   • Coverage: {dense_combinations/possible_combinations*100:.1f}%")

# Context-specific insights
print(f"\n🔍 CONTEXT-SPECIFIC INSIGHTS:")
for context in df_all['context_type'].unique():
    context_data = df_all[df_all['context_type'] == context]
    most_common_dim = context_data.groupby('standardized_dimension')['count'].sum().idxmax()
    most_common_label = context_data.groupby('standardized_label')['count'].sum().idxmax()
    
    print(f"   • {context}:")
    print(f"     - Most common dimension: {most_common_dim}")
    print(f"     - Most common label: {most_common_label}")
    print(f"     - Unique roles: {context_data['role'].nunique()}")

print(f"\n🎯 RECOMMENDATIONS FOR BIAS ANALYSIS:")
print("   1. Focus on roles present in all contexts for comprehensive bias comparison")
print("   2. Pay attention to context-aware vs context-free differences") 
print("   3. Analyze dimension-label combinations with high frequency")
print("   4. Consider cohort-specific patterns within each context")
print("   5. Look for roles with significant variation across contexts")

## Additional Analysis Functions

Here are some utility functions for further analysis:

In [ ]:
def analyze_specific_role(role_name):
    """Analyze a specific role across all contexts"""
    role_data = df_all[df_all['role'] == role_name]
    
    if role_data.empty:
        print(f"Role '{role_name}' not found in data")
        return
    
    print(f"🔍 ANALYSIS FOR ROLE: {role_name.upper()}")
    print("=" * 50)
    
    # Context distribution
    context_dist = role_data.groupby('context_type')['count'].sum()
    print("📊 Data distribution across contexts:")
    for context, count in context_dist.items():
        print(f"   • {context}: {count:,} data points")
    
    # Top dimensions and labels
    top_dims = role_data.groupby('standardized_dimension')['count'].sum().sort_values(ascending=False).head(5)
    top_labels = role_data.groupby('standardized_label')['count'].sum().sort_values(ascending=False).head(5)
    
    print(f"\n📏 Top 5 dimensions for {role_name}:")
    for dim, count in top_dims.items():
        print(f"   • {dim}: {count:,}")
    
    print(f"\n🏷️ Top 5 labels for {role_name}:")
    for label, count in top_labels.items():
        print(f"   • {label}: {count:,}")

def compare_contexts(dimension=None, label=None):
    """Compare contexts for a specific dimension or label"""
    if dimension:
        data = df_all[df_all['standardized_dimension'] == dimension]
        title = f"Dimension: {dimension}"
    elif label:
        data = df_all[df_all['standardized_label'] == label]
        title = f"Label: {label}"
    else:
        print("Please specify either dimension or label")
        return
    
    if data.empty:
        print(f"No data found for {title}")
        return
    
    print(f"🔍 CONTEXT COMPARISON FOR {title.upper()}")
    print("=" * 50)
    
    context_comparison = data.groupby('context_type')['count'].sum().sort_values(ascending=False)
    
    for context, count in context_comparison.items():
        pct = (count / context_comparison.sum()) * 100
        print(f"   • {context}: {count:,} ({pct:.1f}%)")

print("🔧 Additional analysis functions defined:")
print("   • analyze_specific_role(role_name) - Analyze a specific role")
print("   • compare_contexts(dimension='...') - Compare contexts for a dimension")
print("   • compare_contexts(label='...') - Compare contexts for a label")

## Example Usage

Uncomment and run the cells below to see examples of the analysis functions:

In [ ]:
# Example: Analyze a specific role
# analyze_specific_role('ceo')

In [ ]:
# Example: Compare contexts for a specific dimension
# compare_contexts(dimension='type')

In [ ]:
# Example: Compare contexts for a specific label
# compare_contexts(label='medium')